In [1]:
import jinja2
environment = jinja2.Environment()
template = environment.from_string("Hello, {{ name }}!")
template.render(name="World")

'Hello, World!'

In [38]:
base_image = "python:3.8-slim"
build_arg_vars = "HF_REPOS,HF_REPOS"
build_arg_vars = build_arg_vars.split(",")

In [39]:
template_text = """
FROM {{ base_image }} as builder
{% for arg in arg_var %}
ARG {{ arg }}{% endfor %}
{% for arg in arg_var %}
ENV {{ arg }}=${{ arg }}{% endfor %}
RUN apt-get update && apt-get install -y
RUN pip install -r requirements.txt
RUN python3 hf.py

FROM {{ base_image }} as runner
COPY --from=builder /models /models
COPY --from=builder /venv /venv

RUN chmod +x {{ run_script }}
CMD ["./{{ run_script }}"]
"""

In [40]:
template_text = jinja2.Template(template_text)
template_text = template_text.render(
    base_image=base_image,
    arg_var=build_arg_vars,
    run_script="run_runpod.sh"
)
print(template_text)


FROM python:3.8-slim as builder

ARG HF_REPOS
ARG HF_REPOS

ENV HF_REPOS=$HF_REPOS
ENV HF_REPOS=$HF_REPOS
RUN apt-get update && apt-get install -y
RUN pip install -r requirements.txt
RUN python3 hf.py

FROM python:3.8-slim as runner
COPY --from=builder /models /models
COPY --from=builder /venv /venv

RUN chmod +x run_runpod.sh
CMD ["./run_runpod.sh"]


In [29]:
with open("Dockerfile.serverless", "w") as f:
    f.write(template_text)
